# Hanoi EV Charging Dispatch — PyTorch PPO (Kaggle Ready)

Notebook này dùng để chạy train **PPO** trên **Kaggle** (có sẵn PyTorch, CUDA GPU, NumPy, Matplotlib).

### Cách dùng trên Kaggle
1. Upload thư mục `rl_ppo_ev/` (đã gồm sẵn thư mục con `data/` chứa 4 file JSON) thành 1 Kaggle Dataset.
2. Trong Kaggle Notebook: **Add Input** → chọn dataset vừa upload → bấm **Run All** hoặc **Save Version**.

In [ ]:
import os, sys, subprocess
from pathlib import Path
import torch

print('PyTorch:', torch.__version__, '| CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

search_roots = ['/kaggle/input', '/kaggle/working', '.']
CODE_DIR = None
DATA_DIR = None

for root in search_roots:
    if not os.path.exists(root):
        continue
    if CODE_DIR is None:
        for p in Path(root).rglob('train_ppo.py'):
            CODE_DIR = str(p.parent)
            break
    if DATA_DIR is None:
        for p in Path(root).rglob('hanoi_stations.json'):
            DATA_DIR = str(p.parent)
            break
    if CODE_DIR and DATA_DIR:
        break

print('CODE_DIR:', CODE_DIR)
print('DATA_DIR:', DATA_DIR)
assert CODE_DIR is not None, 'Không tìm thấy train_ppo.py — hãy Add Input dataset chứa rl_ppo_ev/'
assert DATA_DIR is not None, 'Không tìm thấy hanoi_stations.json — hãy đảm bảo dataset có thư mục rl_ppo_ev/data/'

In [ ]:
def run_stream(cmd_list):
    print('Running:', ' '.join(cmd_list))
    proc = subprocess.Popen(
        cmd_list,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )
    for line in proc.stdout:
        print(line, end='')
    ret = proc.wait()
    assert ret == 0, f'Command failed with exit code {ret}'

SAVE_DIR = '/kaggle/working/checkpoints'
TOTAL_TIMESTEPS = 150000
NUM_ENVS = 16
NUM_STEPS = 128

run_stream([
    sys.executable, '-u', f'{CODE_DIR}/train_ppo.py',
    '--data-dir', DATA_DIR,
    '--save-dir', SAVE_DIR,
    '--total-timesteps', str(TOTAL_TIMESTEPS),
    '--num-envs', str(NUM_ENVS),
    '--num-steps', str(NUM_STEPS),
    '--eval-every', '10',
    '--eval-episodes', '50',
    '--device', 'auto',
])

In [ ]:
run_stream([
    sys.executable, '-u', f'{CODE_DIR}/eval_ppo.py',
    '--data-dir', DATA_DIR,
    '--checkpoint', f'{SAVE_DIR}/best_ppo.pt',
    '--episodes', '100',
])

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

df = pd.read_csv(f'{SAVE_DIR}/metrics.csv')
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].plot(df['global_step'], df['eval_return'], marker='o')
axes[0].set_title('Eval Mean Return')
axes[0].set_xlabel('Timesteps')
axes[0].grid(True, alpha=0.3)

axes[1].plot(df['global_step'], df['mean_cost_vnd'] / 1000.0, marker='o', color='tab:orange')
axes[1].set_title('Mean Charging Cost (k VND)')
axes[1].set_xlabel('Timesteps')
axes[1].grid(True, alpha=0.3)

axes[2].plot(df['global_step'], df['mean_wait_min'], marker='o', color='tab:green')
axes[2].set_title('Mean Queue Wait (min)')
axes[2].set_xlabel('Timesteps')
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(f'{SAVE_DIR}/learning_curve.png', dpi=150)
plt.show()